# 1. Download and clean the Orbis data, the list of licensed CSPs (DNB) and the offshore leaks (ICIJ)

Replication code for *Uncovering the size of the illegal corporate service provider industry in the Netherlands: a network approach* (Garcia-Bernardo, Witteman & Vlaanderen, EPJ Data Science, 2022).

This notebook covers Sections 3.1.1, 3.1.2 and 3.1.4 and Appendices A.2–A.3 of the paper:

1. (Optional) Download the Orbis exports that Orbis sends by e-mail.
2. Merge the Orbis exports (addresses, legal events, directors, financials and ownership).
3. Clean addresses (including previous addresses, parsed from the legal events), directors, company information and ownership.
4. Create the list of addresses and names of licensed CSPs (DNB register + branches found in Orbis).
5. Read the ICIJ offshore leaks database.

**Inputs** (see README): Orbis exports in `data_raw_orbis/` (not included, licensed data), the DNB register `data_raw_other/Internetregister-WTTTK.csv` (included) and the ICIJ csv files in `data_raw_other/offshoreleaks_*/` (download from ICIJ).

**Outputs**: `data_final/` and `data_processing/` (they contain Orbis data: do not share them).

In [ ]:
import os
import re
import json
import pickle

# Download e-mails
import email
import imaplib
import urllib.request

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_rows', 40)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
pd.options.mode.chained_assignment = None  # ignore SettingWithCopy warnings

for folder in ["data_raw_orbis", "data_raw_other", "data_processing", "data_final/layers", "Figures"]:
    os.makedirs(folder, exist_ok=True)

## 1. Download the Orbis exports (optional)

Orbis e-mails a download link for each (large) export. These functions read all e-mails of the mailbox that received the exports (use a mailbox dedicated to them) and download the files into `data_raw_orbis/`. Skip this section if you saved the exports there manually.

Credentials are read from environment variables. Never write them in the notebook.

In [ ]:
def login_gmail(from_email, from_pass):
    """Log into a gmail inbox."""
    imap = imaplib.IMAP4_SSL("imap.gmail.com", 993)
    imap.login(from_email, from_pass)
    imap.select('INBOX')
    return imap

def get_inbox(imap):
    """Return the download links found in all e-mails of the inbox."""
    url_list = []
    status, datas = imap.search(None, 'ALL')
    for num in datas[0].split():
        status, data = imap.fetch(num, '(RFC822)')
        for response_part in data:
            if isinstance(response_part, tuple):
                msg = email.message_from_bytes(response_part[1])
                # We only need the body with the link (small files are also attached)
                if len(msg.get_payload()) == 2:
                    msg = msg.get_payload()[0]
                body = msg.get_payload(decode=True).decode("utf8")
                url = re.search(r"(?P<url>https?://[^\s]+)", body).group("url")
                url_list.append(url)
    return url_list

In [ ]:
# Only runs when the mailbox credentials are set
if os.environ.get("ORBIS_EMAIL"):
    imap = login_gmail(os.environ["ORBIS_EMAIL"], os.environ["ORBIS_EMAIL_PASSWORD"])
    url_list = get_inbox(imap)

    files_downloaded = os.listdir("data_raw_orbis")
    for url in url_list:
        url_save = url[url.rfind("/")+1:]  # keep the hash as file name
        if url_save in files_downloaded:  # skip files downloaded earlier
            continue
        urllib.request.urlretrieve(url, f"data_raw_orbis/{url_save}")

## 2. Merge the Orbis exports

Each export is an Excel file with a *Results* and a *Search summary* sheet. We exported four types of files (Appendix A.2), which we identify by their columns.

In [ ]:
SEOaddresses, SEOevents, SEOdirectors, SEOothe = [], [], [], []

files_process = os.listdir("data_raw_orbis")
for i, file in enumerate(files_process):
    if file[0] == "~":  # skip temporary files of open Excel files
        continue
    print("{:.1%}".format(i/len(files_process)), end=" | ")

    df = pd.read_excel(f"data_raw_orbis/{file}", sheet_name=None)  # slow but that's okay

    # Keep the update number of the Orbis database (to check that all exports are consistent)
    update_number = df["Search summary"].loc[2, "Unnamed: 1"]
    df = df["Results"]
    df["Update number"] = update_number

    if "National legal form" in df.columns:
        SEOaddresses.append(df)
    elif "Legal events - Date" in df.columns:
        SEOevents.append(df)
    elif "DM\nJob title" in df.columns:
        SEOdirectors.append(df)
    elif "CSH - Name" in df.columns:
        SEOothe.append(df)
    else:
        print(f"File {file} failed")

df_SEOaddresses = pd.concat(SEOaddresses)
df_SEOevents = pd.concat(SEOevents)
df_SEOdirectors = pd.concat(SEOdirectors)
df_SEOothe = pd.concat(SEOothe)
del SEOaddresses, SEOevents, SEOdirectors, SEOothe

In [ ]:
# Data quality checks: number of companies and update numbers of each type of file
for name, df in [("addresses", df_SEOaddresses), ("events", df_SEOevents), ("directors", df_SEOdirectors), ("other", df_SEOothe)]:
    print(name, len(df["BvD ID number"].dropna().unique()), df["Update number"].unique())

In [ ]:
df_SEOaddresses.to_csv("data_processing/SEOaddresses.tsv", sep="\t", index=None)
df_SEOevents.to_csv("data_processing/SEOevents.tsv", sep="\t", index=None)
df_SEOdirectors.to_csv("data_processing/SEOdirectors.tsv", sep="\t", index=None)
df_SEOothe.to_csv("data_processing/SEOothe.tsv", sep="\t", index=None)

## 3. Clean the Orbis data
### 3.1 Addresses

In [ ]:
df_SEOaddresses = pd.read_csv("data_processing/SEOaddresses.tsv", sep="\t", keep_default_na=False, na_values=['n.a.', '-', ''],
                              converters={"VAT/Tax number": str, 'Street number\nLocal Alphabet.1': str, 'Trade register number': str})

# Numbers read as floats ("123.0")
for col in ["VAT/Tax number", 'Street number\nLocal Alphabet.1', 'Trade register number']:
    df_SEOaddresses[col] = df_SEOaddresses[col].str.replace(".0", "", regex=False)

In [ ]:
rename_cols = {'Company name Latin alphabet': "company_name",
               'BvD ID number': "bvd_id",
               'VAT/Tax number': "vat_id",
               'Trade register number': "kvk_id",
               'National legal form': "legal_form",
               'Operating revenue (Turnover)\nth USD Last avail. yr': "operating_revenue",
               'Address line 1\nLocal Alphabet': "address_full",
               'Postcode\nLocal Alphabet': "address_postcode",
               'City\nLocal Alphabet': "address_city",
               'Country ISO code': "address_iso2",
               'Street\nLocal Alphabet': "address_street",
               'Street number\nLocal Alphabet': "address_number",
               'Address line 1\nLocal Alphabet.1': "address_extra_full",
               'Postcode\nLocal Alphabet.1': "address_extra_postcode",
               'City\nLocal Alphabet.1': "address_extra_cit",
               'Country ISO code\nLocal Alphabet': "address_extra_iso2",
               'Street\nLocal Alphabet.1': "address_extra_street",
               'Street number\nLocal Alphabet.1': "address_extra_number",
               'PO Box\nLocal Alphabet': "address_pobox"}
drop_cols = ["Unnamed: 0", "Update number"]
df_SEOaddresses = df_SEOaddresses.rename(columns=rename_cols).drop(columns=drop_cols)
df_SEOaddresses = df_SEOaddresses.drop_duplicates()
df_SEOaddresses.head()

In [ ]:
# Company-level information (merged with other company information later)
constant_variable = ["company_name", "bvd_id", "vat_id", "kvk_id", "legal_form", "operating_revenue"]
df_constant = df_SEOaddresses.loc[:, constant_variable]

In [ ]:
# Current addresses: office address, extra (postal) address and PO box
address_variables1 = ["company_name", "bvd_id", "address_full", "address_postcode", "address_city", "address_iso2", "address_street", "address_number"]
address_variables2 = ["company_name", "bvd_id", "address_extra_full", "address_extra_postcode", "address_extra_iso2", "address_extra_street", "address_extra_number"]
address_variables3 = ["company_name", "bvd_id", "address_pobox"]

dfs_add = []
for l_v in [address_variables1, address_variables2, address_variables3]:
    df_add = df_SEOaddresses.loc[:, l_v]
    df_add.columns = address_variables1[:len(l_v)]

    if "address_full" in l_v:
        df_add["address_type"] = "Office"
    else:
        df_add["address_type"] = "Postal"

    dfs_add.append(df_add.dropna(subset=["address_full"]))

df_add = pd.concat(dfs_add, sort=False)
df_add["address_year"] = 2019
df_add.head()

### 3.2 Legal events (previous names, phone numbers, sectors and addresses)

In [ ]:
df_SEOevents = pd.read_csv("data_processing/SEOevents.tsv", sep="\t", keep_default_na=False, na_values=['n.a.', '-', ''])

rename_cols = {'Company name Latin alphabet': "company_name",
               'BvD ID number': "bvd_id",
               'Legal events - Date': "event_date",
               'Legal events - Description': "event_description",
               'Legal events - Source': "event_source",
               'Legal events - Official identifier': "event_id",
               'Legal events - Type': "event_type",
               'Legal events - Details': "event_details"}
drop_cols = ["Unnamed: 0", "Update number"]
df_SEOevents = df_SEOevents.rename(columns=rename_cols).drop(columns=drop_cols)

# The name and ID of the company are only given in the first row of each company
df_SEOevents["company_name"] = df_SEOevents["company_name"].ffill()
df_SEOevents["bvd_id"] = df_SEOevents["bvd_id"].ffill()

df_SEOevents = df_SEOevents.drop_duplicates()
df_SEOevents.head()

In [ ]:
older_names = df_SEOevents.loc[df_SEOevents["event_description"] == "Change of Name"]
older_phones = df_SEOevents.loc[df_SEOevents["event_description"] == "Change of Phone number"]
older_nace = df_SEOevents.loc[df_SEOevents["event_description"] == "Change of Primary SBI"]

# "Formerly: <value>" -> "<value> (<year>)", joined per company
older_names["name"] = older_names["event_details"].str[10:] + " (" + older_names["event_date"].str[-4:] + ")"
older_names = older_names.groupby("bvd_id")["name"].apply(lambda x: " ||| ".join(x))

older_phones["name"] = older_phones["event_details"].str[10:] + " (" + older_phones["event_date"].str[-4:] + ")"
older_phones = older_phones.groupby("bvd_id")["name"].apply(lambda x: " ||| ".join(x))

older_nace["name"] = older_nace["event_details"].str[14:18] + " (" + older_nace["event_date"].str[-4:] + ")"
older_nace = older_nace.groupby("bvd_id")["name"].apply(lambda x: " ||| ".join(x))

Previous addresses are given as `Formerly: Street Number|Postcode CITY` (Appendix A.3). We split them with the following regular expressions.

In [ ]:
def extract_street(full_add):
    """Split an address line into street and number."""
    if "postbus" in full_add.lower():
        return full_add, ""
    if "p.o. box" in full_add.lower():
        return full_add, ""
    if "po box" in full_add.lower():
        return full_add, ""

    # Dutch address (street followed by number)
    re_exp = r"^(.*?)\s+(\d*\w*(\-|\/)?\d+.*)$"
    match = re.findall(re_exp, full_add)
    if len(match) > 0:
        match = match[0]
        return match[0], match[1]

    # Number followed by street (mostly foreign addresses)
    re_exp = r"^(\d*\w*(\-|\/)?\d+.*?)\s+(.*?)$"
    match = re.findall(re_exp, full_add)
    if len(match) > 0:
        match = match[0]
        return match[-1], match[0]

    return full_add, ""

def extract_postcode(full_add):
    """Split a "postcode city" string into postcode and city."""
    # Dutch postcode
    re_exp = r"\d{4}\s?[A-Z]{2}"
    match = re.findall(re_exp, full_add)
    if len(match) > 0:
        city = full_add[full_add.find(match[0])+len(match[0]):]
        return match[0], city.strip()

    # UK postcode
    re_exp = r"([A-Z][A-HJ-Y]?\d[A-Z\d]? ?\d[A-Z]{2}|GIR ?0A{2})"
    match = re.findall(re_exp, full_add)
    if len(match) > 0:
        city = full_add[:full_add.find(match[0])]
        if len(city) < 3:
            city = full_add[full_add.find(match[0])+len(match[0]):]
        return match[0], city.strip()

    # US postcode (and others)
    re_exp = r"([A-Z]?-?\d{4,5})(-\d{4})*"
    match = re.findall(re_exp, full_add)
    if len(match) > 0:
        match = match[0]
        city = full_add[:full_add.find(match[0])]
        if len(city) < 3:
            city = full_add[full_add.find(match[0])+len(match[0]):]
        return match[0], city.strip()

    return "", full_add

In [ ]:
older_addresses = df_SEOevents.loc[df_SEOevents["event_description"].isin(["Change of Postal address", "Change of Office address"])]

# Delete the "Formerly: " text
older_addresses.loc[:, "address_full"] = older_addresses["event_details"].str[10:]

# Postcode and city (right of "|")
older_addresses.loc[:, "address_other"] = older_addresses["address_full"].apply(lambda x: x[x.find("|")+1:])
older_addresses.loc[:, "address_postcode"], older_addresses.loc[:, "address_city"] = zip(*older_addresses["address_other"].str.strip().apply(extract_postcode))

# Street and number (left of "|")
older_addresses.loc[:, "address_full"] = older_addresses["address_full"].apply(lambda x: x[:x.find("|")])
older_addresses.loc[:, "address_street"], older_addresses.loc[:, "address_number"] = zip(*older_addresses["address_full"].str.replace("- ", "-").apply(extract_street))

older_addresses.loc[:, "address_iso2"] = ""
older_addresses.loc[:, "address_type"] = older_addresses["event_description"].str.split().str[2]
older_addresses.loc[:, "address_year"] = older_addresses["event_date"].str[-4:]
older_addresses = older_addresses.loc[:, address_variables1+["address_type", "address_year"]]

# Delete commas and other non alphanumeric characters from the start of the city
older_addresses["address_city"] = older_addresses["address_city"].str.replace(r"^\W", "", regex=True).str.strip()

### 3.3 Directors

In [ ]:
df_SEOdirectors = pd.read_csv("data_processing/SEOdirectors.tsv", sep="\t", keep_default_na=False, na_values=['n.a.', '-', ''])

rename_cols = {'Company name Latin alphabet': "company_name",
               'BvD ID number': "bvd_id",
               'DM\nUCI (Unique Contact Identifier)': "dir_id",
               'DM\nFull name': "dir_name",
               'DM\nJob title': "dir_title",
               'DM\nCurrent or previous': "dir_status",
               'DM\nCorresponding BvDID (when applicable)': "dir_bvd_id"}
drop_cols = ["Unnamed: 0", "Update number", "company_name"]
df_SEOdirectors = df_SEOdirectors.rename(columns=rename_cols).drop(columns=drop_cols)

# The ID of the company is only given in the first row of each company
df_SEOdirectors["bvd_id"] = df_SEOdirectors["bvd_id"].ffill()

df_SEOdirectors = df_SEOdirectors.drop_duplicates().dropna(subset=["dir_id"])
df_SEOdirectors.head()

In [ ]:
# Fill missing values so we can group by that column
df_SEOdirectors["dir_bvd_id"] = df_SEOdirectors["dir_bvd_id"].fillna("")

# A director can hold several positions in the same company: concatenate them
df_SEOdirectors = df_SEOdirectors.groupby(["bvd_id", "dir_id", "dir_name", "dir_status", "dir_bvd_id"]).agg({"dir_title": lambda x: "; ".join(x)}).reset_index()

In [ ]:
# Remove the KvK number from the beginning of corporate directors' names (e.g. "12345678 EXAMPLE B.V.")
directors_with_kvk = df_SEOdirectors["dir_name"].str.contains(r"^\d{8}\s.{5,}", regex=True)
print(directors_with_kvk.sum())
df_SEOdirectors.loc[directors_with_kvk, "dir_name"] = df_SEOdirectors.loc[directors_with_kvk, "dir_name"].str.extract(r"\d{8}\s(.*)")[0]

df_SEOdirectors["dir_name"] = df_SEOdirectors["dir_name"].str.upper()

### 3.4 Financial information and ownership

In [ ]:
df_SEOothe = pd.read_csv("data_processing/SEOothe.tsv", sep="\t", keep_default_na=False, na_values=['n.a.', '-', ''],
                         converters={"NACE Rev. 2, core code (4 digits)": str})
df_SEOothe["NACE Rev. 2, core code (4 digits)"] = df_SEOothe["NACE Rev. 2, core code (4 digits)"].str.replace(".0", "", regex=False)

rename_cols = {'Company name Latin alphabet': "company_name",
               'BvD ID number': "bvd_id",
               'P/L before tax\nth USD Last avail. yr': "profit_before_tax",
               'Total assets\nth USD Last avail. yr': "total_assets",
               'Shareholders funds\nth USD Last avail. yr': "shareholder_funds",
               'Number of employees\nLast avail. yr': "number_employees",
               'NACE Rev. 2, core code (4 digits)': "nace",
               'CSH - BvD ID number': "csh_bvd_id",
               'CSH - Name': "csh_name",
               'CSH - Country ISO code': "csh_iso2",
               'CSH - Operating revenue (Turnover)\nm USD': "csh_operating_revenue"}
drop_cols = ["Unnamed: 0", "Update number", "company_name"]
df_SEOothe = df_SEOothe.rename(columns=rename_cols).drop(columns=drop_cols)
df_SEOothe = df_SEOothe.drop_duplicates()
df_SEOothe.head()

In [ ]:
# Company-level information
cols_constant = ["bvd_id", "profit_before_tax", "total_assets", "shareholder_funds", "number_employees", "nace"]
df_constant2 = df_SEOothe.loc[:, cols_constant].dropna(subset=['bvd_id'])

# Ownership: global ultimate owner (CSH) of each company
cols_own = ["bvd_id", 'csh_bvd_id', 'csh_name', "csh_iso2", "csh_operating_revenue"]
df_own = df_SEOothe.loc[:, cols_own].dropna(subset=["csh_bvd_id"])
df_own["bvd_id"] = df_own["bvd_id"].ffill()
df_own.head()

### 3.5 Combine everything

Company information (`df_company_info`), current and previous addresses (`df_company_add`), directors (`df_SEOdirectors`) and ownership (`df_own`).

In [ ]:
df_company_info = pd.concat([df_constant.set_index("bvd_id"), df_constant2.set_index("bvd_id"), older_names, older_phones, older_nace], axis=1, sort=False).reset_index()
df_company_info.head(2)

In [ ]:
_NUMBER_SPACE = re.compile(r"([0-9]+(\.[0-9]+)?)")
_NO_SPACE = re.compile(r"\s+")
_ALPHA = re.compile(r"[-\./,'\(\)+\&\":#;_@\*]")

def clean_address(full_add):
    """Normalize an address: spaces around numbers, no punctuation, single spaces, upper case."""
    if isinstance(full_add, float):
        return ""
    match = _NUMBER_SPACE.sub(r" \1 ", full_add).strip()
    match = _ALPHA.sub("", match)
    match = _NO_SPACE.sub(" ", match)
    return match.upper()

In [ ]:
# Concatenate current and previous addresses
df_company_add = pd.concat([df_add, older_addresses], axis=0, sort=False)

# Get the street name and number from the full address when missing
df_company_add.loc[:, "address_street2"], df_company_add.loc[:, "address_number2"] = zip(*df_company_add["address_full"].str.replace("- ", "-").apply(extract_street))
replace_bool = df_company_add["address_street"].fillna("XXX").isin(["", "XXX"])
df_company_add.loc[replace_bool, "address_street"] = df_company_add.loc[replace_bool, "address_street2"]
replace_bool = df_company_add["address_number"].fillna("XXX").isin(["", "XXX"])
df_company_add.loc[replace_bool, "address_number"] = df_company_add.loc[replace_bool, "address_number2"]

# Formatting
df_company_add["address_postcode"] = df_company_add["address_postcode"].str.replace(" ", "").replace("-", "").apply(clean_address)
df_company_add["address_full"] = df_company_add["address_full"].astype(str).apply(clean_address)
df_company_add["address_city"] = df_company_add["address_city"].astype(str).apply(clean_address)
df_company_add["address_street"] = df_company_add["address_street"].astype(str).apply(clean_address)
df_company_add["address_number"] = df_company_add["address_number"].astype(str).str.replace(".0", "", regex=False).apply(clean_address)

df_company_add = df_company_add.drop(columns=["address_number2", "address_street2"])

# Delete spaces (to combine with the list of licensed CSPs)
df_company_add["address_postcode"] = df_company_add["address_postcode"].str.replace(r"\W", "", regex=True)
# Keep the (first) number
df_company_add["address_number"] = df_company_add["address_number"].str.extract(r'(\d+)')

df_company_add.head()

Addresses are identified by postcode and number (`address_combined_gr`). The same address can be written in different ways: within each postcode+number, we cluster similar address strings (character n-gram TF-IDF, single linkage) and map them to the most common spelling.

In [ ]:
vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 4))
vec.fit(list(df_company_add["address_full"].dropna().unique()))

def tfidf(list1, debug=False, th=0.5):
    """Cluster labels of a list of addresses (single linkage on the cosine distance)."""
    len_l = len(list1)
    list1 = vec.transform(list1)
    cos = 1-cosine_similarity(list1, dense_output=True)
    cos[cos < 0] = 0
    if debug:
        print(np.round(cos, 2))
    # Keep the upper triangle (condensed distance matrix)
    cos = cos[np.triu_indices(len_l, k=1)]
    Z = linkage(cos, 'single')
    return fcluster(Z, t=th, criterion='distance')

In [ ]:
add_to_count = df_company_add["address_full"].value_counts().to_dict()

df_company_add["address_combined_gr"] = df_company_add["address_postcode"].fillna("MissingValue") + " | " + df_company_add["address_number"].fillna("MissingValue")
study_addresses = df_company_add.groupby("address_combined_gr").agg({"address_full": "nunique"})
study_addresses.columns = ["count_addresses"]
df_company_add = pd.merge(df_company_add, study_addresses.reset_index(), how="left")
df_company_add.head()

In [ ]:
old_to_new_address = dict()

# Restrict to postcode+number with several spellings, and don't change PO boxes
study_addresses = df_company_add.loc[df_company_add["count_addresses"] > 1]
study_addresses = study_addresses.loc[~study_addresses["address_full"].str.contains(r"POSTBUS|PO\s?BOX")]

for postcode, group in study_addresses.groupby("address_combined_gr"):
    addresses = group["address_full"].unique()
    if len(addresses) < 2:  # we removed PO boxes
        continue
    # Delete the numbers to avoid increasing the similarity if the numbers match each other
    addresses_match = [re.sub(r"\d", "", _).replace("  ", " ").strip() for _ in group["address_full"].unique()]
    labels = tfidf(addresses_match)

    # Map all addresses in a cluster to the address with the highest count (or the shortest one in case of ties)
    current_label = -1
    change_add = []  # addresses to change
    set_ad = ""      # top address
    max_ad = 0       # count of top address

    for l, ad in sorted(zip(labels, addresses)):
        # New cluster: save the previous one and reset the markers
        if l != current_label:
            for ad_change in change_add:
                old_to_new_address[ad_change] = set_ad

            current_label = l
            change_add = [ad]
            max_ad = add_to_count[ad]
            set_ad = ad
        # Same cluster: check if the new address is better
        else:
            change_add.append(ad)
            test_max_ad = add_to_count[ad]
            if test_max_ad > max_ad:
                max_ad = test_max_ad
                set_ad = ad
            elif test_max_ad == max_ad:
                if len(ad) < len(set_ad):
                    max_ad = test_max_ad
                    set_ad = ad

    # The last cluster
    for ad_change in change_add:
        old_to_new_address[ad_change] = set_ad

json.dump(old_to_new_address, open("data_final/old_to_new_address.json", "w+"))

In [ ]:
df_company_add["address_full_norm"] = df_company_add["address_full"].map(old_to_new_address)
add_bool = df_company_add["address_full_norm"].fillna("MissingValue") == "MissingValue"
df_company_add.loc[add_bool, "address_full_norm"] = df_company_add.loc[add_bool, "address_full"]

In [ ]:
df_company_info.to_csv("data_final/df_company_info.tsv", sep="\t", index=None)
df_company_add.to_csv("data_final/df_addresses.tsv", sep="\t", index=None)
df_SEOdirectors.to_csv("data_final/df_directors.tsv", sep="\t", index=None)
df_own.to_csv("data_final/df_ownership.tsv", sep="\t", index=None)

In [ ]:
# Global ultimate owner (GUO) of each company
df_own = pd.read_csv("data_final/df_ownership.tsv", sep="\t")
bvd2guo = df_own.drop_duplicates(subset=["bvd_id"]).set_index("bvd_id").to_dict()["csh_bvd_id"]
json.dump(bvd2guo, open("data_final/bvd2guo.json", "w+"))

## 4. Licensed CSPs (Section 3.1.2)

Addresses and names of the licensed CSPs, from the DNB public register (https://www.dnb.nl/en/public-register/), plus the addresses of their branches found in Orbis.

In [ ]:
old_to_new_address = json.load(open("data_final/old_to_new_address.json", "r"))
df_company_add = pd.read_csv("data_final/df_addresses.tsv", sep="\t")

kvk2bvd = df_SEOaddresses.set_index("kvk_id")["bvd_id"].to_dict()
bvd2kvk = df_SEOaddresses.set_index("bvd_id")["kvk_id"].to_dict()

In [ ]:
# Orbis IDs of the licensed CSPs, matched using the KvK numbers in the DNB register (5 completed by hand).
# There are more IDs than KvK numbers because of branches (we want those!)
list_trusts = ["NL33144202", "NL33126512", "NL33258621", "NL320154920001T", "NL33036603", "NL33136491", "NL33282777", "NL33203015", "NL34179384", "NL33263803", "NL000010540318", "NL000016279972", "NL000016578740", "NL000017210933", "NL000017814545", "NL000017814596", "NL000018731937", "NL000019841523", "NL000019841531", "NL000028240391", "NL09129526", "NL14053998", "NL14055055", "NL14082438", "NL17048517", "NL17072014", "NL17138025", "NL17231584", "NL17234593", "NL18079208", "NL20077238", "NL20087541", "NL20147040", "NL21016442", "NL22035259", "NL24122367", "NL24162847", "NL24191163", "NL24196052", "NL24196667", "NL24275916", "NL24278780", "NL24288896", "NL24292157", "NL24310904", "NL24370676", "NL24425583", "NL24478862", "NL24484591", "NL24486594", "NL27123680", "NL27123949", "NL27128025", "NL27130346", "NL27157648", "NL27168222", "NL27174166", "NL27183802", "NL27216361", "NL27223479", "NL27235317", "NL27270869", "NL28065255", "NL28092999", "NL28096268", "NL28110197", "NL30091120", "NL30100157", "NL30145617", "NL30176226", "NL30209156", "NL32015492", "NL32059602", "NL32081192", "NL32082214", "NL32086251", "NL32094108", "NL32094141", "NL32142197", "NL33000302", "NL33001955", "NL33058755", "NL33075510", "NL33093266", "NL33135957", "NL33143523", "NL33144784", "NL33151567", "NL33160097", "NL33161897", "NL33163838", "NL33165094", "NL33165748", "NL33168549", "NL33190865", "NL33191144", "NL33191879", "NL33195052", "NL33202790", "NL33210270", "NL33217636", "NL33226415", "NL33230767", "NL33246011", "NL33270765", "NL33272809", "NL33274430", "NL33276574", "NL33277207", "NL33278110", "NL33284969", "NL33286854", "NL33289126", "NL33291692", "NL33294508", "NL33294521", "NL33297030", "NL34043466", "NL34088149", "NL34092518", "NL34093368", "NL34098061", "NL34117808", "NL34125346", "NL34128584", "NL34147143", "NL34151781", "NL34158682", "NL34177688", "NL34179971", "NL34184038", "NL34203884", "NL34206371", "NL34208584", "NL34213515", "NL34234797", "NL34243465", "NL34245837", "NL34267742", "NL34269697", "NL34270994", "NL34279318", "NL34280199", "NL34282323", "NL34285874", "NL34292344", "NL34310762", "NL34314274", "NL34317555", "NL34318880", "NL34319829", "NL34319830", "NL34323854", "NL34324886", "NL34332832", "NL34355413", "NL34355425", "NL37112959", "NL37134298", "NL50776460", "NL51276747", "NL51906414", "NL52430855", "NL52489159", "NL52552810", "NL52763161", "NL52887227", "NL53024362", "NL53503988", "NL53723848", "NL55057640", "NL55057764", "NL55148271", "NL55150268", "NL56100582", "NL56611161", "NL57250243", "NL58684948", "NL60468521", "NL60683406", "NL61016942", "NL61537608", "NL62370006", "NL62686739", "NL66088771", "NL66088844", "NL67549608", "NL68482833", "NL70257396", "NL74309579", "NL75654768", "NL000003006395", "NL000003903281", "NL000005110068", "NL000008355592", "NL000009423842", "NL000010022643", "NL000011260831", "NL000012923036", "NL000016279956", "NL000017194989", "NL000017566363", "NL000018570364", "NL000018990282", "NL000019242913", "NL000019979207", "NL000020581696", "NL000022360131", "NL000022556524", "NL000023713305", "NL000023781688", "NL000024324965", "NL000024433403", "NL000024744840", "NL000025565435", "NL000025998137", "NL000026146401", "NL000026503816", "NL000027679470", "NL000029089670", "NL000030242460", "NL000031097006", "NL000031684041", "NL000032750080", "NL000032770316", "NL000043877036", "NL241223670001T", "NL280652550001T", "NL301001570001T", "NL30183298", "NL302091560001T", "NL330366030001T", "NL330366030002T", "NL331265120001T", "NL331265120004T", "NL331600970002T", "NL331618970002T", "NL331685490002T", "NL33191041", "NL332030150003T", "NL332765740001T", "NL342135150000T", "NL34253291", "NL342709940001T", "NL342823230001T", "NL54435218"]
trust_addresses = df_company_add.loc[df_company_add["bvd_id"].isin(list_trusts)]
trust_addresses["kvk_id"] = trust_addresses["bvd_id"].map(bvd2kvk)
trust_addresses["official"] = 0
trust_addresses.head(2)

In [ ]:
# Addresses in the DNB register.
# The data rows of this csv end with ";", so pandas uses the first column as index and every header
# refers to the column to its right. The column selection below accounts for this shift.
official_addresses = pd.read_csv("data_raw_other/Internetregister-WTTTK.csv", sep=";", encoding="latin1")[["Deelregister", "APF Statutaire_Naam", "Relatienummer", "Datum in liquidatie", "NEN-Landcode postadres", "Handelsnaam", "Adres vestiging", "Postcode vestiging", "Land vestiging", "NEN-Landcode vestiging", "Postadres", "Plaats postadres", "Land postadres"]]
official_addresses.columns = ["deelregister", "related_number", "company_name", "other_names", "kvk_id", "address_full", "address_postcode", "address_city", "address_iso2", "address_full_p", "address_city_p", "address_postcode_p", "address_iso2_p"]
official_addresses1 = official_addresses.loc[:, ["deelregister", "related_number", "company_name", "other_names", "kvk_id", "address_full", "address_postcode", "address_city", "address_iso2"]]
official_addresses1["adddress_type"] = "Office"
official_addresses2 = official_addresses.loc[:, ["deelregister", "related_number", "company_name", "other_names", "kvk_id", "address_full_p", "address_postcode_p", "address_city_p", "address_iso2_p"]]
official_addresses2["adddress_type"] = "Postal"
official_addresses2.columns = official_addresses1.columns

In [ ]:
# The register has no postcodes for the postal addresses: get them with the Google geocoder (Section 3.1.2).
# The API key is read from an environment variable.
from geopy import geocoders

g = geocoders.GoogleV3(api_key=os.environ["GOOGLE_API_KEY"])

postcodes = []
addresses = (official_addresses2["address_full"] + ", " + official_addresses2["address_city"] + ", " + official_addresses2["address_postcode"]).values
for i, address in enumerate(addresses):
    if isinstance(address, float) or ("Postbus" in address):
        postcodes.append("")
    else:
        location = g.geocode(address, timeout=10)
        postcodes.append(extract_postcode(location.address)[0])

official_addresses2["address_postcode"] = postcodes

In [ ]:
official_addresses = pd.concat([official_addresses1, official_addresses2], sort=False)
official_addresses = official_addresses.dropna(subset=["address_full"])
official_addresses.loc[:, "address_street"], official_addresses.loc[:, "address_number"] = zip(*official_addresses["address_full"].str.replace("- ", "-").apply(extract_street))
official_addresses["bvd_id"] = official_addresses["kvk_id"].astype(str).map(kvk2bvd)
official_addresses["kvk_id"] = official_addresses["kvk_id"].astype(str)

official_addresses["company_name"] = official_addresses["company_name"].str.upper()
official_addresses["other_names"] = official_addresses["other_names"].str.upper()
official_addresses["address_full"] = official_addresses["address_full"].str.upper()
official_addresses["address_street"] = official_addresses["address_street"].str.upper()
# Delete spaces (to combine with the Orbis addresses)
official_addresses["address_postcode"] = official_addresses["address_postcode"].str.replace(r"\W", "", regex=True)
# Keep the (first) number
official_addresses["address_number"] = official_addresses["address_number"].str.extract(r'(\d+)')

official_addresses["address_combined_gr"] = official_addresses["address_postcode"].fillna("MissingValue") + " | " + official_addresses["address_number"].fillna("MissingValue")

# CSPs that are part of a group get the ID of the group
official_addresses.loc[official_addresses["related_number"].fillna("XXX") != "XXX", "deelregister"] = official_addresses.loc[official_addresses["related_number"].fillna("XXX") != "XXX", "related_number"]
kvk2dnb_id = official_addresses.set_index("kvk_id")["deelregister"].to_dict()

del official_addresses["related_number"]
official_addresses["official"] = 1

In [ ]:
# One row for each trade name
rows_add = []
for r, row in official_addresses.iterrows():
    if row["company_name"] == row["other_names"]:
        continue

    row_save = row.copy()
    if isinstance(row["other_names"], str):
        for name in row["other_names"].split(", "):
            if name != row["company_name"]:
                row_save["company_name"] = name
                rows_add.append(row_save)
rows_add = pd.concat(rows_add, axis=1).T

official_addresses = pd.concat([official_addresses, rows_add])
del official_addresses["other_names"]

In [ ]:
# Merge the addresses from the DNB register and from Orbis
df_trust_addresses = pd.concat([trust_addresses, official_addresses], sort=False, ignore_index=True)
df_trust_addresses["deelregister"] = df_trust_addresses["kvk_id"].astype(str).map(kvk2dnb_id)
df_trust_addresses["address_full_norm"] = df_trust_addresses["address_full"].replace(old_to_new_address)
df_trust_addresses = df_trust_addresses.dropna(subset=["deelregister"])

df_trust_addresses.to_csv("data_final/df_trust_addresses.tsv", sep="\t", index=None)

## 5. Offshore leaks (Section 3.1.4)

Names of entities, officers and intermediaries, and addresses, in the ICIJ offshore leaks database. Each leak must be in its own folder in `data_raw_other/` with "offshore" in the folder name (as in the original csv download).

In [ ]:
adds, ents, offs, ints = [], [], [], []

for folder in os.listdir("data_raw_other/"):
    if "offshore" in folder:
        for file in os.listdir(f"data_raw_other/{folder}"):
            if "address" in file:
                adds += pd.read_csv(f"data_raw_other/{folder}/{file}", usecols=["address"])["address"].tolist()
            elif "entity" in file:
                ents += pd.read_csv(f"data_raw_other/{folder}/{file}", usecols=["name"])["name"].tolist()
            elif "intermediary" in file:
                ints += pd.read_csv(f"data_raw_other/{folder}/{file}", usecols=["name"])["name"].tolist()
            elif "officer" in file:
                offs += pd.read_csv(f"data_raw_other/{folder}/{file}", usecols=["name"])["name"].tolist()
            else:
                print("Not used:", file)

print(len(ints), "intermediaries,", len(offs), "officers,", len(ents), "entities,", len(adds), "addresses")

pickle.dump(adds, open("data_final/icij_addresses.dump", "wb+"))
pickle.dump(ents, open("data_final/icij_entities.dump", "wb+"))
pickle.dump(offs, open("data_final/icij_officers.dump", "wb+"))
pickle.dump(ints, open("data_final/icij_intermediaries.dump", "wb+"))